## Import

In [6]:
import warnings
warnings.filterwarnings("ignore")
import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torchvision as tv
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:5'

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Dataset

In [8]:
from datasets.Student_t import MultivariateStudentT

d = 32
dim_x = dim_y = d//2
rho = 0.8
df = 3
mean = np.zeros(dim_x + dim_y)
V = np.eye(dim_x)*rho
dispersion = np.eye(dim_x + dim_y)
dispersion[0:dim_x, :][:, dim_x:] = V
dispersion[dim_x:, :][:, 0:dim_x] = V


dataset = MultivariateStudentT(
        dim_x=dim_x,
        dim_y=dim_y,
        mean=mean,
        dispersion=dispersion,
        df=df)

X, Y = dataset.sample(10000)
X, Y = torch.Tensor(X).float().to(device), torch.Tensor(Y).float().to(device)

print(f'MI is {dataset.mutual_information(): 0.2f}')
print('X', X.shape, 'Y', Y.shape)

MI is  8.87
X torch.Size([10000, 16]) Y torch.Size([10000, 16])


## Hyperaparams

In [9]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.n_bridges = 4
        self.wd = 1e-5
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]
architecture_encode = [d//2, 200, 200, d//2]

## MI estimate

In [12]:
## Mutual information neural diffusion estimate (MINDE)
from estimators.MINDE import MINDE

hyperparams.t_patience = 500
hyperparams.dim = d//2
hyperparams.device = device
hyperparams.importance_sampling = True

estimator = MINDE(None, None, None, hyperparams)

estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 1.506677508354187 loss val= 1.5004668235778809 best val loss= 1.5004668235778809 best t= 0
finished: t= 76 loss= 0.7808837294578552 loss val= 0.7517228126525879 best val loss= 0.5246542692184448 best t= 57
finished: t= 152 loss= 0.7137386798858643 loss val= 0.7963800430297852 best val loss= 0.4924525022506714 best t= 140
finished: t= 228 loss= 0.793176531791687 loss val= 0.6465252041816711 best val loss= 0.4924525022506714 best t= 140
finished: t= 304 loss= 0.6956690549850464 loss val= 0.6064227819442749 best val loss= 0.48828697204589844 best t= 302
finished: t= 380 loss= 0.6872997879981995 loss val= 0.507592499256134 best val loss= 0.48828697204589844 best t= 302
finished: t= 456 loss= 0.5380384922027588 loss val= 0.5120853185653687 best val loss= 0.48185569047927856 best t= 436
finished: t= 532 loss= 0.49632614850997925 loss val= 0.5956509709358215 best val loss= 0.48185569047927856 best t= 436
finished: t= 608 loss= 0.5177540183067322 loss val= 0.76308798789978

In [10]:
## Mutual information neural estimate (MINE)
from estimators.MINE import MINE

estimator = MINE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.0341765396296978 loss val= 0.040148355066776276 best val loss= 0.040148355066776276 best t= 0
finished: t= 76 loss= -5.433551788330078 loss val= -5.775294780731201 best val loss= -5.866919994354248 best t= 71
finished: t= 152 loss= -5.493594646453857 loss val= -5.874919891357422 best val loss= -5.884188652038574 best t= 139
finished: t= 228 loss= -5.599976539611816 loss val= -5.907525539398193 best val loss= -5.953574180603027 best t= 169
finished: t= 304 loss= -5.5462470054626465 loss val= -5.769047260284424 best val loss= -5.953574180603027 best t= 169
finished: t= 380 loss= -5.586939811706543 loss val= -5.610563278198242 best val loss= -6.0097503662109375 best t= 339
finished: t= 456 loss= -5.429418087005615 loss val= -5.7053022384643555 best val loss= -6.0097503662109375 best t= 339


true MI: 8.873405423659612
est MI: 6.842170715332031


In [15]:
## Vector copula estimation
from estimators.VCE import VCE

estimator = VCE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))


K components= 5
nde type: FM
finished: t= 0 loss= 3.526508092880249 loss val= 4.26127815246582 best val loss= 4.26127815246582 best t= 0
finished: t= 126 loss= 4.945468425750732 loss val= 2.9630751609802246 best val loss= 2.5981531143188477 best t= 60
finished: t= 252 loss= 2.3043053150177 loss val= 2.8743371963500977 best val loss= 2.516897201538086 best t= 210


finished: t= 0 loss= 3.8000099658966064 loss val= 4.161556720733643 best val loss= 4.161556720733643 best t= 0
finished: t= 126 loss= 2.559452772140503 loss val= 2.946216106414795 best val loss= 2.6989190578460693 best t= 41


finished: t= 0 loss= 123.09208679199219 loss val= 140.10838317871094 best val loss= 140.10838317871094 best t= 0
finished: t= 101 loss= 36.37990951538086 loss val= 39.13502502441406 best val loss= 39.048343658447266 best t= 56
finished: t= 202 loss= 36.5858154296875 loss val= 39.24275207519531 best val loss= 39.048343658447266 best t= 56


K= 5 time used: 116.2188286781311
true MI: 8.873405423659612
est